# Cải tiến SEED3-MEAN bằng cách tăng số nhánh XGBoost

`SEED3-MEAN` gộp ba nhánh `XGB-D03` bằng trung bình xác suất. Thử thêm hai nhánh để xem việc đổi seed có cải thiện khả năng xếp hạng giao dịch gian lận hay không. Tất cả nhánh dùng cùng dữ liệu cuộc thi và đặc trưng của `XGB-D03`. Mô hình, huấn luyện, suy luận và tổ hợp dùng XGBoost cùng các phép tính số, không có LLM hay đầu ra từ bên ngoài dự án.

Notebook chạy với kernel của môi trường local `s6e11_gpu/.build/venv`. Cần có `train.csv`, `test.csv`, `sample_submission.csv` và các dự đoán đối chứng trong `s6e11_gpu/plan_runs/results/full`. Để đọc kết quả đã lưu, chạy các cell với `TRAIN_OR_RESUME = False`.

## 1. Nạp trình chạy và tìm dữ liệu

In [1]:
import sys
from dataclasses import asdict
from math import isclose
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents]
candidates = [*roots, *(root / "Predicting Credit Card Fraud" for root in roots)]
relative_runner = Path("s6e11_gpu/seed3_improvement/run_seed_bagging.py")
project = next((root for root in candidates if (root / relative_runner).is_file()), None)
if project is None:
    raise FileNotFoundError("Mở notebook trong thư mục dự án Predicting Credit Card Fraud.")
runner_dir = (project / relative_runner).parent
if str(runner_dir) not in sys.path:
    sys.path.insert(0, str(runner_dir))

import run_seed_bagging as bags

print(f"Thư mục dự án  {project}")
print(f"Thư mục kết quả  {bags.OUTPUT}")

Thư mục dự án  D:\UET\ML\Predicting Credit Card Fraud
Thư mục kết quả  D:\UET\ML\Predicting Credit Card Fraud\s6e11_gpu\seed3_improvement\results


## 2. Kiểm tra đối chứng trên đúng năm fold

Một dự đoán ngoài fold (out-of-fold, OOF) được tạo bởi nhánh chưa học hàng giao dịch đó. Mỗi fold ngoài có tập huấn luyện riêng. Trình chạy chọn số cây bằng một tập dừng sớm ở bên trong phần huấn luyện, sau đó huấn luyện lại trên toàn bộ phần huấn luyện của fold. Nó dự đoán fold giữ lại và `test.csv`. Các điểm test được lấy trung bình qua năm fold.

Cell này kiểm tra phiên bản dữ liệu, ID, fold của các dự đoán và tái tạo trung bình ba nhánh gốc. AUC trong bảng là AUC tính một lần trên toàn bộ dự đoán OOF đã ghép.

In [2]:
run = bags.PlanRun(bags.OUTPUT)
original, baseline, base_metrics = bags.load_baseline(run)
baseline_rows = []
for name in bags.ORIGINAL:
    info = bags.read_json(bags.REFERENCE / name / "metrics.json")
    baseline_rows.append({
        "Nhánh": name,
        "Seed cây": info["config"]["model_seed"],
        "Seed encoder": info["config"]["encoder_seed"],
        "OOF ROC AUC": info["oof_auc"],
    })
display(pd.DataFrame(baseline_rows).style.format({"OOF ROC AUC": "{:.9f}"}))
print(f"{bags.BASELINE}  {base_metrics['oof_auc']:.9f} OOF ROC AUC")
print(f"{run.manifest['rows']:,} dòng train · {run.manifest['test_rows']:,} dòng test · {len(run.partitions)} fold")

699,960 train, 299,984 test, 5 folds | D:\UET\ML\Predicting Credit Card Fraud\s6e11_gpu\seed3_improvement\results
SEED3-MEAN  0.950269278 OOF ROC AUC
699,960 dòng train · 299,984 dòng test · 5 fold


,Nhánh,Seed cây,Seed encoder,OOF ROC AUC
0,XGB-D03,42,2026,0.950182303
1,XGB-D03-S71,71,2026,0.950074149
2,XGB-D03-S93,42,2028,0.950091415


## 3. Hai cấu hình mới và các tổ hợp

`XGB-D03-S71-E2028` kết hợp seed cây 71 và seed encoder 2028 vốn đã xuất hiện riêng trong hai nhánh gốc. `XGB-D03-S103-E2032` dùng cặp seed mới. Seed cây chi phối phần lấy mẫu ngẫu nhiên của XGBoost. Seed encoder chi phối cách chia fold bên trong khi tạo mã hóa theo nhãn (target encoding). Hai nhánh giữ nguyên nhóm đặc trưng, tham số cây và các fold ngoài của đối chứng.

In [3]:
specs = bags.load_experiment()
display(pd.DataFrame([
    {"Nhánh": spec.name, "Seed cây": spec.model_seed, "Seed encoder": spec.encoder_seed}
    for spec in specs
]))
display(pd.DataFrame([
    {"Tổ hợp": name, "Số nhánh": len(members), "Trọng số mỗi nhánh": f"1/{len(members)}",
     "Thành viên": ", ".join(members)}
    for name, members in {bags.BASELINE: bags.ORIGINAL, **bags.BAGS}.items()
]))

,Nhánh,Seed cây,Seed encoder
0,XGB-D03-S71-E2028,71,2028
1,XGB-D03-S103-E2032,103,2032


,Tổ hợp,Số nhánh,Trọng số mỗi nhánh,Thành viên
0,SEED3-MEAN,3,1/3,"XGB-D03, XGB-D03-S71, XGB-D03-S93"
1,SEED4-CROSS-MEAN,4,1/4,"XGB-D03, XGB-D03-S71, XGB-D03-S93, XGB-D03-S71..."
2,SEED4-NEW-MEAN,4,1/4,"XGB-D03, XGB-D03-S71, XGB-D03-S93, XGB-D03-S10..."
3,SEED5-MEAN,5,1/5,"XGB-D03, XGB-D03-S71, XGB-D03-S93, XGB-D03-S71..."


## 4. Huấn luyện hoặc tiếp tục từ checkpoint

Đổi biến bên dưới thành `True` để huấn luyện lại các fold còn thiếu trên RTX 4060 rồi chấm tổ hợp. Mỗi fold đã lưu đúng cấu hình và ID được nạp lại từ `results`. Để tránh hai phiên ghi cùng thư mục, chờ lệnh `run_seed_bagging.py` đang chạy trong terminal kết thúc trước khi bật cell huấn luyện này.

In [4]:
TRAIN_OR_RESUME = False

if TRAIN_OR_RESUME:
    for spec in specs:
        run.run(spec)
else:
    print("Đang dùng chế độ đọc kết quả đã lưu.")

Đang dùng chế độ đọc kết quả đã lưu.


## 5. Đọc kết quả và đối chiếu theo fold

Tổ hợp được chấm bằng ROC AUC và average precision (AP), tức thước đo chú trọng khả năng tìm các giao dịch gian lận. `Δ AUC` được tính so với `SEED3-MEAN` trên cùng dữ liệu. Khoảng bootstrap ghép cặp 95% lấy mẫu lại cùng tập hàng cho cả hai mô hình để đánh giá độ dao động của chênh lệch.

In [5]:
def load_completed_members():
    results = []
    for spec in specs:
        directory = run.output / spec.name
        metrics_path = directory / "metrics.json"
        if not metrics_path.is_file() or not (directory / "predictions.npz").is_file():
            return None
        info = bags.read_json(metrics_path)
        config = asdict(spec)
        config["groups"] = list(config["groups"])
        if (info["status"] != "completed" or info["config"] != config
                or any(info.get(key) != value for key, value in run.manifest.items())):
            raise ValueError(f"Kết quả lưu không khớp cấu hình hoặc dữ liệu của {spec.name}.")
        run.load(spec.name)
        results.append(info)
    return results


def load_current_evaluation(member_metrics):
    path = run.output / "evaluation.json"
    if member_metrics is None or not path.is_file():
        return None
    saved = bags.read_json(path)
    if saved.get("baseline") != base_metrics or saved.get("new_members") != member_metrics:
        return None
    candidates = {row["name"]: row for row in saved.get("candidates", [])}
    if set(candidates) != set(bags.BAGS) or saved.get("selected") not in {bags.BASELINE, *bags.BAGS}:
        return None
    for name, members in bags.BAGS.items():
        path = run.output / name / "metrics.json"
        if not path.is_file() or not path.with_name("predictions.npz").is_file():
            return None
        metrics = bags.read_json(path)
        row = candidates[name]
        if (row["members"] != list(members)
                or metrics["config"]["weights"] != [1 / len(members)] * len(members)
                or any(row[key] != metrics[key] for key in ("oof_auc", "average_precision", "log_loss"))
                or any(metrics.get(key) != value for key, value in run.manifest.items())):
            return None
        pooled_gain = metrics["oof_auc"] - base_metrics["oof_auc"]
        new_folds = {fold["fold"]: fold["auc"] for fold in metrics["folds"]}
        base_folds = {fold["fold"]: fold["auc"] for fold in base_metrics["folds"]}
        if new_folds.keys() != base_folds.keys() or len(row["delta_by_fold"]) != len(base_folds):
            return None
        fold_gains = [new_folds[fold] - base_folds[fold] for fold in sorted(base_folds)]
        if (not isclose(row["delta_oof_auc"], pooled_gain, rel_tol=0, abs_tol=1e-12)
                or any(not isclose(saved_gain, fold_gain, rel_tol=0, abs_tol=1e-12)
                       for saved_gain, fold_gain in zip(row["delta_by_fold"], fold_gains, strict=True))):
            return None
        run.load(name)
    return saved


member_metrics = load_completed_members()
evaluation = load_current_evaluation(member_metrics)
if evaluation is None and TRAIN_OR_RESUME:
    evaluation = bags.score_bags(run, original, baseline, base_metrics)

if member_metrics is not None:
    display(pd.DataFrame([
        {"Nhánh mới": row["name"], "OOF ROC AUC": row["oof_auc"],
         "AP": row["average_precision"], "Thiết bị": row["folds"][0]["device"]}
        for row in member_metrics
    ]).style.format({"OOF ROC AUC": "{:.9f}", "AP": "{:.9f}"}))

,Nhánh mới,OOF ROC AUC,AP,Thiết bị
0,XGB-D03-S71-E2028,0.950200547,0.528347145,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
1,XGB-D03-S103-E2032,0.950121476,0.528453895,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
if evaluation is None:
    print("Chưa có bảng đánh giá đầy đủ cho cấu hình hiện tại. Chạy lại hai cell ở mục 5 sau khi lượt GPU hoàn tất.")
else:
    result_rows = [{
        "Tổ hợp": bags.BASELINE,
        "OOF ROC AUC": base_metrics["oof_auc"],
        "Δ AUC": 0.0,
        "AP": base_metrics["average_precision"],
        "Log-loss": base_metrics["log_loss"],
        "Fold tăng": "—",
        "Khoảng 95% của Δ AUC": "—",
    }]
    fold_rows = []
    for row in evaluation["candidates"]:
        interval = row["bootstrap"]
        result_rows.append({
            "Tổ hợp": row["name"], "OOF ROC AUC": row["oof_auc"], "Δ AUC": row["delta_oof_auc"],
            "AP": row["average_precision"], "Log-loss": row["log_loss"],
            "Fold tăng": f"{row['folds_up']}/{len(run.partitions)}",
            "Khoảng 95% của Δ AUC": f"[{interval['ci_low']:+.9f}, {interval['ci_high']:+.9f}]",
        })
        fold_rows.append({
            "Tổ hợp": row["name"],
            **{f"Fold {index}": gain for index, gain in enumerate(row["delta_by_fold"], start=1)},
        })
    display(pd.DataFrame(result_rows).style.format({
        "OOF ROC AUC": "{:.9f}", "Δ AUC": "{:+.9f}", "AP": "{:.9f}", "Log-loss": "{:.9f}",
    }))
    display(pd.DataFrame(fold_rows).style.format({
        f"Fold {index}": "{:+.9f}" for index in range(1, len(run.partitions) + 1)
    }))

,Tổ hợp,OOF ROC AUC,Δ AUC,AP,Log-loss,Fold tăng,Khoảng 95% của Δ AUC
0,SEED3-MEAN,0.950269278,+0.000000000,0.528975397,0.044409259,—,—
1,SEED4-CROSS-MEAN,0.950303108,+0.000033831,0.529076182,0.044399258,5/5,"[-0.000001014, +0.000068749]"
2,SEED4-NEW-MEAN,0.950309071,+0.000039793,0.529264892,0.044391143,4/5,"[-0.000003160, +0.000082377]"
3,SEED5-MEAN,0.950331140,+0.000061862,0.529295983,0.044385929,5/5,"[+0.000015747, +0.000108718]"


,Tổ hợp,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5
0,SEED4-CROSS-MEAN,+0.000003095,+0.000018739,+0.000051537,+0.000019386,+0.000082764
1,SEED4-NEW-MEAN,+0.000024022,+0.000061831,-0.000017206,+0.000107105,+0.000055819
2,SEED5-MEAN,+0.000027237,+0.000063478,+0.000031297,+0.000104802,+0.000113762


## 6. Đọc quyết định

`SEED5-MEAN` là ứng viên chính. Tiêu chí thay đối chứng gồm tăng ít nhất **0,000100 OOF ROC AUC**, tăng ở **ít nhất 4/5 fold** và cận dưới của khoảng bootstrap 95% **lớn hơn 0**. Nếu nhiều tổ hợp đạt cả ba điều kiện, trình chạy chọn tổ hợp có OOF ROC AUC cao nhất.

Các kết quả ở đây dựa trên năm fold của tập train. Chúng chưa có điểm Kaggle Public hoặc Private Leaderboard.

In [7]:
if evaluation is not None:
    selected = evaluation["selected"]
    eligible = [row["name"] for row in evaluation["candidates"] if row["preferred"]]
    if eligible:
        print("Các tổ hợp đạt tiêu chí  " + ", ".join(eligible))
        print(f"Chọn {selected}.")
    else:
        print(f"Giữ {selected}. Chưa có tổ hợp nào đạt cả ba điều kiện.")

Giữ SEED3-MEAN. Chưa có tổ hợp nào đạt cả ba điều kiện.
